# BLACKBOX D6 — SAM2 tracking upgrade (Colab)

**Owner: Pranav. CSRT (`blackbox/pipeline/track.py`, D4) is the guaranteed path — this notebook is the GPU upgrade, only worth running if it's working well before Gate 1 (15:30). If it isn't, skip it; nothing downstream depends on it.**

This notebook must write a `tracks.json` that matches the exact same contract
`track.py` writes (see `blackbox/schemas.py`: `Tracks` / `TrackFrame`) — same
fields, same units (floor metres, origin at one corner of the 48 ft box), same
gap rule (a non-wide frame is present with `pos: null`, never interpolated
across a shot boundary). Any downstream module (telemetry, events, overlay)
should not be able to tell which tracker produced the file.

Shape of the run:
1. Upload the fight clip + `shots.json` (+ `calibration.json`).
2. Click-prompt both bots on the first wide frame of each wide segment.
3. SAM2-small mask propagation through that segment.
4. Mask centroids -> homography -> floor metres.
5. Download `tracks.json`, drop it into `data/processed/<fight_id>/tracks.json` locally.

Runtime: **Runtime > Change runtime type > GPU (T4 is enough)**.

## 1. Install SAM2

This installs Meta's Segment Anything 2 and downloads the **small** checkpoint
(good speed/accuracy trade-off for a laptop-free Colab session). This cell is
the single most likely failure point — if `pip install` or the checkpoint
download breaks, that's the signal to abandon this notebook and stick with
D4 (CSRT); the fixture-first fallback is not a rewrite.

In [ ]:
!pip install -q git+https://github.com/facebookresearch/sam2.git
!mkdir -p /content/checkpoints
!wget -q -P /content/checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
!wget -q -O /content/sam2.1_hiera_s.yaml https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2/configs/sam2.1/sam2.1_hiera_s.yaml
print('SAM2 installed.')

## 2. Upload inputs

Upload three files from your local `data/processed/<fight_id>/` and
`data/frames/<fight_id>/` on the laptop that ran D1-D3:
* the cut clip — `clip.mp4`
* `shots.json` (D2 output)
* `calibration.json` (D3 output)

In [ ]:
from google.colab import files

print('Select clip.mp4, shots.json, calibration.json (multi-select).')
uploaded = files.upload()
assert 'clip.mp4' in uploaded, 'need clip.mp4'
assert 'shots.json' in uploaded, 'need shots.json'
assert 'calibration.json' in uploaded, 'need calibration.json'

FIGHT_ID = input('fight_id (matches data/processed/<fight_id>/): ').strip()

In [ ]:
import json

with open('shots.json') as f:
    shots = json.load(f)
with open('calibration.json') as f:
    calibration = json.load(f)

FPS = 10  # matches D1 (blackbox/pipeline/ingest.py TRACK_FPS) and schemas.Tracks.fps
BOT_NAMES = input('bot names, comma-separated, same order as meta.json bots: ').split(',')
BOT_NAMES = [b.strip() for b in BOT_NAMES]
print('bots:', BOT_NAMES)
print(f"{len(shots['shots'])} shots, {sum(s['wide'] for s in shots['shots'])} wide")

## 3. Extract frames per wide segment

Frame *i* is always at `t = i / FPS`, 0-indexed — same convention as D1's
`ingest.py` (`-start_number 0`). SAM2's video predictor wants one JPEG per
frame in its own directory.

In [ ]:
import subprocess, pathlib

FRAMES_DIR = pathlib.Path('/content/frames')
FRAMES_DIR.mkdir(exist_ok=True)
subprocess.run(
    ['ffmpeg', '-y', '-i', 'clip.mp4', '-vf', f'fps={FPS}', '-start_number', '0',
     str(FRAMES_DIR / 'frame_%06d.jpg')],
    check=True,
)
all_frames = sorted(FRAMES_DIR.glob('*.jpg'))
print(f'{len(all_frames)} frames extracted at {FPS} fps')

## 4. Click-prompt each wide segment

Colab has no `cv2.imshow`/mouse-callback GUI, so this uses the boring, always-
works version: the first frame of each wide segment is shown with a labelled
50 px grid, and you type in the (x, y) pixel coordinate of each bot. Slower
than a click, but nothing to debug at 19:00.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np


def show_with_grid(frame_path, step=50):
    img = cv2.cvtColor(cv2.imread(str(frame_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(10, 10 * h / w))
    ax.imshow(img)
    ax.set_xticks(range(0, w, step))
    ax.set_yticks(range(0, h, step))
    ax.grid(color='yellow', alpha=0.4, linewidth=0.5)
    plt.show()


def prompt_points(frame_path, bot_names):
    show_with_grid(frame_path)
    points = {}
    for bot in bot_names:
        raw = input(f'{bot} pixel coords as "x,y": ')
        x, y = (float(v) for v in raw.split(','))
        points[bot] = (x, y)
    return points

## 5. SAM2 propagation per wide segment

Each wide shot from `shots.json` gets its own predictor session (SAM2 tracks
within one continuous clip; a camera cut is exactly the kind of discontinuity
D4 also refuses to track across, so this mirrors that gap rule structurally,
not just in the output).

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    '/content/sam2.1_hiera_s.yaml', '/content/checkpoints/sam2.1_hiera_small.pt'
)


def mask_centroid(mask: np.ndarray) -> tuple[float, float] | None:
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return float(xs.mean()), float(ys.mean())

In [ ]:
import shutil

segment_centroids: dict[int, dict[str, tuple[float, float] | None]] = {}  # frame index -> bot -> (x, y) px

for shot in shots['shots']:
    if not shot['wide']:
        continue
    start_i, end_i = int(shot['start_t'] * FPS), min(len(all_frames), int(shot['end_t'] * FPS))
    if end_i - start_i < 1:
        continue

    seg_dir = pathlib.Path('/content/segment')
    if seg_dir.exists():
        shutil.rmtree(seg_dir)
    seg_dir.mkdir()
    for j, src in enumerate(all_frames[start_i:end_i]):
        shutil.copy(src, seg_dir / f'{j:06d}.jpg')

    points = prompt_points(all_frames[start_i], BOT_NAMES)

    state = predictor.init_state(video_path=str(seg_dir))
    predictor.reset_state(state)
    for obj_id, bot in enumerate(BOT_NAMES):
        px, py = points[bot]
        predictor.add_new_points_or_box(
            inference_state=state, frame_idx=0, obj_id=obj_id,
            points=np.array([[px, py]], dtype=np.float32),
            labels=np.array([1], dtype=np.int32),
        )

    for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
        centroids = {}
        for obj_id, logits in zip(obj_ids, mask_logits):
            mask = (logits > 0.0).squeeze().cpu().numpy()
            centroids[BOT_NAMES[obj_id]] = mask_centroid(mask)
        segment_centroids[start_i + frame_idx] = centroids

    print(f"segment {shot['start_t']:.1f}-{shot['end_t']:.1f}s done ({end_i - start_i} frames)")

## 6. Pixel centroids -> floor metres -> tracks.json

Same homography convention as `blackbox/pipeline/calibrate.py`:
`calibration.json`'s `homography` maps **image pixels -> floor metres**.

In [ ]:
H = np.array(calibration['homography'])


def apply_homography(H: np.ndarray, x: float, y: float) -> list[float]:
    v = H @ np.array([x, y, 1.0])
    return [round(float(v[0] / v[2]), 3), round(float(v[1] / v[2]), 3)]


def is_wide(t: float) -> bool:
    return any(s['start_t'] <= t < s['end_t'] and s['wide'] for s in shots['shots'])


frames_out = []
n_wide_tracked = 0
for i in range(len(all_frames)):
    t = round(i / FPS, 2)
    wide = is_wide(t)
    pos = None
    if wide and i in segment_centroids:
        pos = {}
        for bot, px in segment_centroids[i].items():
            if px is not None:
                pos[bot] = apply_homography(H, *px)
        if not pos:
            pos = None
    if wide and pos is not None and len(pos) == len(BOT_NAMES):
        n_wide_tracked += 1
    frames_out.append({'t': t, 'wide': wide, 'pos': pos})

tracks = {
    'fight_id': FIGHT_ID,
    'fps': FPS,
    'coverage': round(n_wide_tracked / len(frames_out), 3) if frames_out else 0.0,
    'frames': frames_out,
}

with open('tracks.json', 'w') as f:
    json.dump(tracks, f, indent=2)
print(f"coverage {tracks['coverage']:.1%}")

## 7. Download

Drop the downloaded file at `data/processed/<fight_id>/tracks.json` on the
laptop, overwriting D4's CSRT output if this run looks better on `--review`.

In [ ]:
files.download('tracks.json')